In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
import pandas as pd

In [2]:
# Load environment variables from .env file
load_dotenv("../private_data/.env")

host = os.getenv("HOST")
db = os.getenv("DB")
port = os.getenv("PORT")
role = os.getenv("ROLE")
pw = os.getenv("PASSWORD")
engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

In [ ]:
# Define your column lists
# cols_deals = ['updated_at', 'deleted_at', 'id', 'legacy_partner_id', 'title', 'description', 'creators_requirement', 'hash_tags', 'status', 'deal_value', 'go_live_at', 'live_until', 'deal_type',
#               'images', 'accepts_international', 'accepted_countries', 'social_requirement_type_id', 'product_name', 'schedule_type', 'schedule_model', 'legacy_id', 'tags', 'gender', 'featured_image', 'company_id', 'partner_id']
# cols_comp = ['applicants_applications_count', 'cancelled_applications_count', 'company_locations', 'completed_applications_count', 'content_types', 'deal_created_at', 'deal_deleted_at', 'deal_id', 'deal_tags', 'deal_updated_at',
#              'first_application_at', 'last_application_at', 'live_since', 'main_image', 'min_social_media_followers', 'pending_applications_count', 'planned_applications_count', 'rejected_applications_count', 'total_company_locations']

# 1. Load Deals
query_deals = f"SELECT * FROM public.deals;"
df_deals = pd.read_sql(query_deals, engine)

# 2. Load Deals Computed with built-in date parsing
# We add deal_id::text as deal_id_str directly in the SQL string
# query_comp = f"""
#     SELECT {', '.join(cols_comp)}, deal_id::text AS deal_id_str
#     FROM public.deals_computed;
# """

query_comp = f"SELECT * FROM public.deals_computed;"

df_deals_comp = pd.read_sql(
    query_comp,
    engine,
    parse_dates=['first_application_at', 'last_application_at']
)

deals_comp_cols = ['applicants_applications_count', 'content_types', 'deal_id', 'main_image', 'min_social_media_followers',
                   'deal_tags', 'live_since', 'first_application_at', 'last_application_at', 'company_locations']
# Merge
df_deals = pd.merge(df_deals_comp[deals_comp_cols],
                    df_deals, left_on='deal_id', right_on='id')

# df_deals = pd.merge(df_deals,
#                     df_deals_comp,
#                     left_on='id', right_on='deal_id', how='left')

del df_deals_comp
# Text takes up a lot of memory, can drop since it is not needed
df_deals.drop(columns=['id'], inplace=True)

# df_deals = df_deals.add_suffix('_deals')

## Cast columns to saveable datatypes

In [ ]:

import json
df_deals["first_application_at"] = pd.to_datetime(
    df_deals["first_application_at"], errors='coerce')
df_deals["last_application_at"] = pd.to_datetime(
    df_deals["last_application_at"], errors='coerce')


df_deals['deal_id'] = df_deals['deal_id'].astype(str)
# df_deals['id'] = df_deals['id'].astype(str)
df_deals['company_id'] = df_deals['company_id'].astype(str)
df_deals['partner_id'] = df_deals['partner_id'].astype(str)


def safe_json_dump(x):
    # Handle NaNs or None
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    # Convert list/dict to valid JSON string
    return json.dumps(x)


# Apply JSON serialization
df_deals['company_locations'] = df_deals['company_locations'].apply(
    safe_json_dump)

In [2]:
import pandas as pd

In [ ]:
BARTER_DEALS_CLEAN

In [4]:
pd.read_parquet('../data/interim/BARTER_DEALS_CLEAN.parquet')

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,company_id,partner_id,first_live_at,cumulative_exposure_hours,actual_exposure_hours,log_exposure,diff_created_deleted,text_word_count,title_word_count,requirements_word_count
1,23,"[{'id': 37, 'name': 'Luxury', 'slug': 'Sketch-...",019c9ef4-ec74-00c8-8361-4d1d1c94daf1,uploads/deals/019c9eb4-23e4-ffff-3493-f77855f8...,1500,None,2026-02-27 12:01:02.467884,2026-02-27 12:25:46.194315,2026-03-11 12:27:02.540406,"[{""id"": ""01KJFAWFX805FS0SN8S8E25F63"", ""name"": ...",...,019c9eae-3f7f-015e-35dd-f2d48a1886af,019c9e91-f47d-0065-dfa1-269ee81133c3,2026-02-27 12:01:02.563772+00:00,168.0,168.0,5.123964,NaT,2,4,3
2,320,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019a4faa-ef7b-00c8-20f8-4a82d4a34ca6,uploads/deals/019a4fa9-937a-ffff-8b77-eae47deb...,2500,None,2025-11-04 16:19:54.042020,2025-11-04 16:39:11.217064,2026-02-23 19:00:51.190445,None,...,None,01995ce0-84c1-0065-9ced-114e35d44e4a,2025-11-04 16:19:54.092772+00:00,168.0,168.0,5.123964,NaT,74,7,85
3,19,"[{'id': 31, 'name': 'Lifestyle', 'slug': 'Mart...",019bbbf9-335f-00c8-5877-5320e658a0ef,uploads/deals/019bbbf9-3402-ffff-d249-5dc4644e...,2500,None,2026-01-14 10:10:29.853087,2026-01-16 20:52:41.703137,2026-02-19 15:16:04.598323,"[{""id"": ""01KERHVYQT05FPFSVYE44E6GG1"", ""name"": ...",...,019bb11d-f384-015e-c396-0d286dfe29a8,01992995-27fe-0065-4f9e-db8ec1e66a27,2026-01-14 10:10:29.898750+00:00,168.0,168.0,5.123964,NaT,109,7,40
6,16,"[{'id': 37, 'name': 'Luxury', 'slug': 'Sketch-...",0197d900-498c-00c8-a245-fcadd548cd71,uploads/deals/0197d8ff-da0b-ffff-c6a5-5e8cfede...,1500,None,2025-07-05 05:12:44.415992,2025-07-05 09:23:34.314057,2025-08-28 07:52:04.715070,None,...,None,01992996-a0cd-0065-2d8f-e1cff4344607,2025-07-05 05:12:44.461110+00:00,168.0,168.0,5.123964,NaT,46,3,110
8,24,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip...",019d06a8-b4f3-00c8-f769-5ca4533be2bc,uploads/deals/019d06a8-5691-ffff-fad3-edf4dec4...,1500,None,2026-03-19 15:13:33.030503,2026-03-19 15:14:41.777785,2026-03-23 22:18:51.559602,None,...,019d06a1-1452-015e-d921-523ccf17ee61,019d0695-ec0e-0065-7fa5-f25475b80d60,2026-03-19 15:13:33.064763+00:00,168.0,168.0,5.123964,NaT,64,9,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7961,77,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019ab5e3-bce4-00c8-7003-59f8812a5bec,uploads/deals/019ab5e2-bab6-ffff-0858-4f337b9c...,5000,None,2025-11-27 09:06:40.042067,2025-11-27 09:27:52.540355,2026-03-06 09:45:25.197404,None,...,None,019a772c-64ae-0065-557f-c151cd2cf6cd,2025-11-27 09:06:40.085751+00:00,168.0,168.0,5.123964,NaT,42,4,106
7962,18,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019b2790-b5be-00c8-ca64-494ae6953920,uploads/deals/019b278f-a047-ffff-d659-00c50908...,2500,None,2025-12-16 14:29:13.942543,2025-12-16 18:45:17.429415,2026-03-17 12:16:28.541502,None,...,None,019a591b-05f1-0065-d625-8dc51440b01a,2025-12-16 14:29:14.002183+00:00,168.0,168.0,5.123964,NaT,60,11,135
7963,9,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019b46da-5536-00c8-d8bb-7e4a28a8b5ef,uploads/deals/019b46d9-d7c6-ffff-08c9-5a2b7be6...,1500,None,2025-12-22 16:17:52.569756,2025-12-22 16:32:50.313069,2025-12-28 02:09:16.427513,None,...,None,019af80b-c799-0065-940f-08f9e637eeef,2025-12-22 16:17:52.611022+00:00,168.0,168.0,5.123964,NaT,38,9,8
7964,3,"[{'id': 19, 'name': 'Sport', 'slug': 'Soccer-B...",019ac090-0f05-00c8-df57-aa9932492da1,uploads/deals/019ac08f-bb30-ffff-e5f5-0436766a...,5000,None,2025-12-03 08:46:03.219421,2025-12-06 09:32:28.684333,2025-12-12 12:26:34.975296,None,...,None,019a9772-4e9b-0065-dae7-921205e27140,2025-12-03 08:46:03.266456+00:00,168.0,168.0,5.123964,NaT,62,4,39


In [3]:
pd.read_parquet('../data/raw/BARTER_DEALS.parquet')

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,social_requirement_type_id,product_name,schedule_type,schedule_model,legacy_id,tags,gender,featured_image,company_id,partner_id
0,7,"[{'id': 38, 'name': 'Music', 'slug': 'Music-No...",019689e3-09ed-00c8-ada6-052b1041b584,uploads/deals/019689e3-0a14-ffff-1130-d620d8c2...,2500,None,2023-09-21 07:40:56.211091,2023-09-24 21:36:08.063257,2023-10-05 17:01:53.664059,"[{""id"": ""01KERHSC8805FW4D6D5GG90JMF"", ""name"": ...",...,2.0,,physical_specific_days_all_day_schedule,"{'date_schedule_end_at': {'day': 8.0, 'hours':...",172.0,None,None,None,019bb11c-b0d3-015e-baaf-b9fa5b347794,01992995-1769-0065-49aa-6a67b7f7af7d
1,23,"[{'id': 37, 'name': 'Luxury', 'slug': 'Sketch-...",019c9ef4-ec74-00c8-8361-4d1d1c94daf1,uploads/deals/019c9eb4-23e4-ffff-3493-f77855f8...,1500,None,2026-02-27 12:01:02.467884,2026-02-27 12:25:46.194315,2026-03-11 12:27:02.540406,"[{""id"": ""01KJFAWFX805FS0SN8S8E25F63"", ""name"": ...",...,1.0,https://webshop.banketbakkerijsmeets.nl/assort...,physical_specific_days_specific_time_schedule,"{'date_schedule_end_at': {'day': 13.0, 'hours'...",NaN,None,unisex,uploads/deals/featured/019c9ef4-92c7-ffff-7841...,019c9eae-3f7f-015e-35dd-f2d48a1886af,019c9e91-f47d-0065-dfa1-269ee81133c3
2,320,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019a4faa-ef7b-00c8-20f8-4a82d4a34ca6,uploads/deals/019a4fa9-937a-ffff-8b77-eae47deb...,2500,None,2025-11-04 16:19:54.042020,2025-11-04 16:39:11.217064,2026-02-23 19:00:51.190445,None,...,2.0,https://getgoyu.com/products/matcha-groene-the...,online_indefinitely_schedule,"{'date_schedule_end_at': None, 'date_schedule_...",NaN,None,None,None,None,01995ce0-84c1-0065-9ced-114e35d44e4a
3,19,"[{'id': 31, 'name': 'Lifestyle', 'slug': 'Mart...",019bbbf9-335f-00c8-5877-5320e658a0ef,uploads/deals/019bbbf9-3402-ffff-d249-5dc4644e...,2500,None,2026-01-14 10:10:29.853087,2026-01-16 20:52:41.703137,2026-02-19 15:16:04.598323,"[{""id"": ""01KERHVYQT05FPFSVYE44E6GG1"", ""name"": ...",...,2.0,https://www.bacardi.com/nl/nl/,online_indefinitely_schedule,"{'date_schedule_end_at': None, 'date_schedule_...",NaN,None,unisex,uploads/deals/featured/019bbbfb-185d-ffff-f220...,019bb11d-f384-015e-c396-0d286dfe29a8,01992995-27fe-0065-4f9e-db8ec1e66a27
4,0,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019689e5-f489-00c8-8c26-06b3ea0961a5,uploads/deals/019689e5-f54b-ffff-dbed-637e9d2a...,5000,None,2023-11-28 14:59:51.626780,NaT,NaT,None,...,3.0,,online_indefinitely_schedule,"{'date_schedule_end_at': None, 'date_schedule_...",487.0,None,None,None,None,01992995-2f32-0065-d8da-ac763ad1f309
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7962,18,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019b2790-b5be-00c8-ca64-494ae6953920,uploads/deals/019b278f-a047-ffff-d659-00c50908...,2500,None,2025-12-16 14:29:13.942543,2025-12-16 18:45:17.429415,2026-03-17 12:16:28.541502,None,...,2.0,www.albayshop.de,online_indefinitely_schedule,"{'date_schedule_end_at': None, 'date_schedule_...",NaN,None,None,None,None,019a591b-05f1-0065-d625-8dc51440b01a
7963,9,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019b46da-5536-00c8-d8bb-7e4a28a8b5ef,uploads/deals/019b46d9-d7c6-ffff-08c9-5a2b7be6...,1500,None,2025-12-22 16:17:52.569756,2025-12-22 16:32:50.313069,2025-12-28 02:09:16.427513,None,...,1.0,https://wattvogel-shop.de/products/damen-hoodi...,online_indefinitely_schedule,"{'date_schedule_end_at': None, 'date_schedule_...",NaN,None,None,None,None,019af80b-c799-0065-940f-08f9e637eeef
7964,3,"[{'id': 19, 'name': 'Sport', 'slug': 'Soccer-B...",019ac090-0f05-00c8-df57-aa9932492da1,uploads/deals/019ac08f-bb30-ffff-e5f5-0436766a...,5000,None,2025-12-03 08:46:03.219421,2025-12-06 09:32:28.684333,2025-12-12 12:26:34.975296,None,...,3.0,https://rehall.com/products/almduft-r-wintersp...,online_indefinitely_schedule,"{'date_sched

In [6]:
df_deals.to_parquet('../data/raw/BARTER_DEALS.parquet', engine='pyarrow')